# Variably saturated flow (VSF): putting Richards into the valley

The synthetic valley so far treats the soil above the water table in a simple way. In the [`synthetic-valley-a-base`](synthetic-valley-a-base.ipynb) model recharge lands straight on the water table, and in the [`synthetic-valley-b-advanced`](synthetic-valley-b-advanced.ipynb) model the **UZF** package routes it through the unsaturated zone with a kinematic-wave approximation. The **VSF** package replaces that with **Richards' equation**: the pressure head is solved in every cell, so the soil has a moisture profile, a capillary fringe, and a lag between recharge and the water table.

This notebook asks a practical question: **what does it take to put the VSF package into the valley model we already have, and what is different?** We start from the base valley (recharge, river, coastal general-head boundary, and wells) because it is the simplest model to change, and work through the steps one at a time.

> **Package reference.** VSF is the Richards-based unsaturated-zone flow package for MODFLOW 6, added through `flopy.mf6.ModflowGwfvsf`. The soil-characteristic curves use the van Genuchten (1980) parameters.

## What this notebook covers

By the end of this notebook you will be able to:

- see why the **vertical grid** has to resolve the unsaturated zone, and what VSF does if it does not,
- hold the **lake**, which the valley builds from very permeable cells, at the base model's level,
- **refine** the top layer of the valley and re-attach the boundaries and wells to the new layers,
- list what the **VSF package** needs: cells flagged for Richards flow, soil parameters, non-convertible cells, Newton, and a robust solver, and
- check how much the answer depends on the **cell thickness** and on the **time step**.

## Imports

Import FloPy, Matplotlib, and NumPy. `vsf_helpers` holds the van Genuchten soil data and curves, `load_valley` that loads the base valley, `lake_stage` and `set_lake` that hold the lake, `refine_top_layer` that refines the grid and moves everything onto it, a reader for the water table, and `aquifer_budget` that runs ZONEBUDGET.

In [ ]:
%matplotlib inline
import pathlib as pl
import time

import flopy
import matplotlib.pyplot as plt
import numpy as np
import vsf_helpers as vh
from vsf_helpers import SOILS

width = 17.15 / 2.54  # figure width, inches, as in the base notebook

## Step 0 &mdash; The starting point: the base valley

`load_valley()` loads the base valley model and sets its workspace. It leaves out evapotranspiration, the seepage drains on the land surface, and the observation packages, which are not needed here. Recharge, the stream (RIV), the coast (GHB), and the two pumping wells stay.

Run it as it is. The model is saturated only: recharge goes straight into the cells of layer 1 and the water table is the head in that layer. This is the reference the VSF models are compared with. The base model takes one time step per yearly stress period; `set_substeps()` splits each yearly period into `nstp` steps (the steady-state period keeps one), because Richards' equation needs smaller steps. Every model here uses the same steps, so only the physics differs.

In [ ]:
nstp = 4  # time steps per yearly stress period, for every model below

base_ws = pl.Path("models/synthetic-valley-vsf-base")
sim_base, gwf_base = vh.load_valley(base_ws)
vh.set_substeps(sim_base, nstp)


def run(sim):
    """Write and run a simulation, and return the run time in seconds."""
    sim.write_simulation(silent=True)
    t0 = time.time()
    success, buff = sim.run_simulation(silent=True)
    assert success, "MODFLOW 6 did not terminate normally:\n" + "\n".join(buff[-20:])
    return time.time() - t0


t_base = run(sim_base)
print(f"base model: {t_base:4.1f} s")

### Where is the unsaturated zone?

VSF needs cells in the unsaturated zone. Compare the thickness of layer 1 with the depth from the land surface to the water table at the end of the steady-state period. A negative depth means the head is above the land surface; without the seepage drains, heads can rise above it.

In [ ]:
dis = gwf_base.dis
top = dis.top.array
botm = dis.botm.array
idomain = dis.idomain.array
thickness1 = top - botm[0]

# end of each stress period, in days
period_ends = np.cumsum([p[0] for p in sim_base.tdis.perioddata.get_data()])
_, wt_base = vh.read_water_tables(base_ws, vsf=False, times=period_ends)
depth_to_wt = top - wt_base[0]

with flopy.plot.styles.USGSMap():
    fig, axd = plt.subplot_mosaic(
        [["A", "B"]], figsize=(width, 0.55 * width), layout="constrained"
    )
    for letter, array, label in [
        ("A", thickness1, "Thickness of layer 1 (m)"),
        ("B", depth_to_wt, "Depth to the water table (m)"),
    ]:
        ax = axd[letter]
        pmv = flopy.plot.PlotMapView(model=gwf_base, ax=ax)
        pc = pmv.plot_array(array)
        fig.colorbar(pc, ax=ax, shrink=0.8, label=label)
        ax.set_aspect("equal")
    flopy.plot.styles.heading(ax=axd["A"], letter="A", heading="Layer 1 thickness")
    flopy.plot.styles.heading(ax=axd["B"], letter="B", heading="Depth to water table")

depth = depth_to_wt.compressed()
print(
    f"layer 1 is {thickness1.min():.1f} to {thickness1.max():.1f} m thick; "
    f"depth to the water table: median {np.median(depth):.2f} m, "
    f"5-95% {np.percentile(depth, 5):.2f} to {np.percentile(depth, 95):.2f} m"
)

**What to look for.** Layer 1 is a single cell 3.6 to 21 m thick (A), thinnest in the south and thickest under the high ground across the middle of the valley. The water table (B) is at or above the land surface in the lake in the northwest, 2 to 4 m below it over most of the south, 6 to 10 m below it in the northeast, and up to 14 m below it under the high ground. In most of the valley the unsaturated zone is therefore **inside one cell**. Richards' equation solves one pressure head per cell, so this grid cannot resolve a moisture profile. We first see what VSF does with it, then fix it.

## Step 1 &mdash; The VSF package

Adding VSF to a model takes four things, whatever the grid:

1. **The package.** VSF is switched on for the cells (`ivarsat`), with the soil model and its parameters: `porosity`, the residual saturation `satres`, and for van Genuchten `alphavgn` and `nvgn`. It also saves the pressure head and the saturation next to the head file.
2. **Non-convertible cells.** VSF replaces the convertible-cell (water table) logic of NPF, so `icelltype` and `iconvert` are 0. The soil curves supply the storage above the water table, and specific yield is no longer used.
3. **Newton.** Richards' equation is strongly nonlinear. The base valley already uses the Newton formulation with under-relaxation, and the VSF models keep it. VSF also runs without Newton; its `storage_scheme` and `kr_averaging` options apply only then. Under Newton, the flow between two cells uses the relative permeability of the upstream cell, the one with the higher head.
4. **A robust solver.** The `COMPLEX` setting, more iterations, and **backtracking**, which shortens a Newton step that makes the residual worse.

The package reads one set of soil curves. The curves give the saturation and the relative permeability as a function of the pressure head, and are what makes the unsaturated zone different from the saturated zone. VSF takes the saturated hydraulic conductivity from NPF, so the curves shape the unsaturated zone, but the conductivity stays the valley's (16 to 133 m/d in layer 1) rather than the soil's; the `ks` values in `SOILS` are for reference only.

In [ ]:
soil_name = "loam"  # try "sand" or "clay"
psi = -np.linspace(0.0, 3.0, 300)  # pressure head, m (drier downward on the plot)

with flopy.plot.styles.USGSPlot():
    fig, axd = plt.subplot_mosaic(
        [["A", "B"]], figsize=(width, 0.45 * width), layout="constrained"
    )
    for name, soil in SOILS.items():
        lw = 2.0 if name == soil_name else 0.8
        axd["A"].plot(vh.van_genuchten_saturation(psi, soil), -psi, lw=lw, label=name)
        axd["B"].plot(vh.van_genuchten_kr(psi, soil), -psi, lw=lw, label=name)
    for letter, label in [("A", "Saturation (-)"), ("B", "Relative permeability (-)")]:
        axd[letter].set_ylim(3.0, 0.0)
        flopy.plot.styles.xlabel(ax=axd[letter], label=label)
    flopy.plot.styles.ylabel(ax=axd["A"], label="Suction -ψ (m)")
    flopy.plot.styles.heading(ax=axd["A"], letter="A", heading="Saturation")
    flopy.plot.styles.heading(ax=axd["B"], letter="B", heading="Relative permeability")
    flopy.plot.styles.graph_legend(ax=axd["B"], loc="center right")

The curves are drawn for three standard soils; the thick line is the soil used in this notebook. Just above the water table (suction near 0) every soil is nearly saturated and conducts at its full conductivity. That near-saturated band is the **capillary fringe**. Moving up, the soil dries and the relative permeability falls much faster than the saturation.

`add_vsf()` makes these changes to a model. This is all the VSF-specific input there is.

In [ ]:
def add_vsf(gwf, soil_name="loam"):
    """Switch on Richards flow in a loaded groundwater flow model."""
    soil = SOILS[soil_name]

    # 2. non-convertible cells: VSF supplies the storage above the water table
    gwf.npf.icelltype.set_data(0)
    gwf.sto.iconvert.set_data(0)

    # 1. the VSF package, with the soil curves and the extra output files
    flopy.mf6.ModflowGwfvsf(
        gwf,
        ivarsat=1,
        soil_model="VanGenuchten",
        porosity=soil["porosity"],
        satres=soil["satres"],
        alphavgn=soil["alpha"],
        nvgn=soil["n"],
        phead_filerecord="sv.phd",
        sat_filerecord="sv.sat",
    )
    gwf.oc.saverecord.set_data({0: [("HEAD", "ALL"), ("BUDGET", "ALL")]})

    # 3. Newton with under-relaxation, as in the base valley
    gwf.name_file.newtonoptions = "NEWTON UNDER_RELAXATION"

    # 4. a robust solver for the nonlinear problem
    ims = next(
        p
        for p in gwf.simulation.sim_package_list
        if isinstance(p, flopy.mf6.ModflowIms)
    )
    ims.complexity = "COMPLEX"
    ims.outer_maximum = 500
    ims.inner_maximum = 300
    ims.outer_dvclose = 1e-6
    ims.inner_dvclose = 1e-7
    ims.linear_acceleration = "BICGSTAB"
    ims.under_relaxation = "DBD"
    ims.backtracking_number = 20
    ims.backtracking_tolerance = 1.1
    ims.backtracking_reduction_factor = 0.3
    ims.backtracking_residual_limit = 100.0

### The lake

The lake in the northwest is not a lake package. The base valley builds it from 153 cells of layer 1 with a hydraulic conductivity of about 600,000 m/d, a vertical conductivity about 2 million times larger still, and a specific yield of 1, so the cells act as open water with a flat surface. VSF would treat those cells as loam.

Left free on the refined grid, the lake stands about 11 cm higher under VSF than in the base model. Water leaves the lake sideways into the soil next to it and drains down from there. Where that soil is unsaturated, VSF multiplies its vertical conductance by the relative permeability, which is close to 0, while NPF keeps the same vertical conductance however wet the cell is. The water backs up beside the lake and the lake rises.

The lake is open water, not soil, so hold it at the base model's level with constant heads (CHD) in every lake cell. `vh.lake_stage()` reads the lake level at the end of every time step of the base run, and `vh.set_lake()` adds the constant heads with that level as a time series. The level never falls more than 12 cm below the top of the lake cells, so the lake cells stay saturated and VSF changes nothing in them. The cost is that the lake no longer responds to VSF.

In [ ]:
lake_times, lake_stages = vh.lake_stage(base_ws, gwf_base)
print(
    f"lake level in the base model: {lake_stages.min():.2f} to {lake_stages.max():.2f} m; "
    f"top of the lake cells: {top[vh.lake_cells(gwf_base)[0]].max():.2f} m"
)

### Add VSF to the native grid

First do the minimum: add VSF to the base valley with its grid unchanged, and the same `nstp` time steps per year as the base model. We use 4 (set above) and test that choice at the end.

In [ ]:
native_ws = pl.Path("models/synthetic-valley-vsf-native")
sim_native, gwf_native = vh.load_valley(native_ws)
vh.set_substeps(sim_native, nstp)
add_vsf(gwf_native, soil_name)
vh.set_lake(gwf_native, lake_times, lake_stages)
t_native = run(sim_native)
print(f"VSF on the native grid: {t_native:4.1f} s")

## Step 2 &mdash; Refine the grid in the vertical

Now give the unsaturated zone cells. `refine_top_layer()` splits the top `refine_depth` meters of layer 1 into sublayers of about `target_dz` meters. One transition cell keeps the rest of layer 1 down to its original bottom, and the layers below are unchanged. The water table is less than 8 m deep in nearly 90% of the valley, so we refine the top 8 m in 1 m cells: 8 sublayers, so the model has 8 more layers and runs in about 15 seconds. Where layer 1 is thinner than 8 m, the sublayers stop at its bottom and are thinner than 1 m. The sublayers follow the land surface, so a cell can sit beside a different sublayer in the next column: about 13% of the horizontal connections in the refined layers join cells that do not overlap at all. MODFLOW 6 connects layer to layer regardless, the usual approximation for sloping layers; here the sublayers slope less than 1.3% at 95% of the connections, and at most about 5%.

Changing the layers affects everything that is defined per layer. This is the part of adding VSF that is most work, and it is the same for any model:

| Package | What changes |
|---|---|
| DIS | layer 1 becomes `n_ref` thin sublayers and a transition cell; the layers below move down by `n_ref` |
| IC, NPF, STO | arrays are copied from layer 1 to every new cell, so each sublayer has the same horizontal and vertical conductivity as layer 1 |
| RCH | no change: recharge still goes to the top cell, and now has to cross the unsaturated zone |
| RIV, GHB | the cell moves to the sublayer that contains the stage or head, so the boundary exchanges water with the aquifer at that elevation. VSF cells never go dry, so a boundary left in the top cell would still work, but it would move water in and out of the soil near the land surface |
| WEL | the layer numbers shift by `n_ref` |
| CHD (lake) | `set_lake()` is called after the refinement, so the lake is held in every new sublayer |

In [ ]:
target_dz = 1.0  # m; smaller resolves the unsaturated zone better but runs slower
refine_depth = 8.0  # m of layer 1, measured from the land surface

refined_ws = pl.Path("models/synthetic-valley-vsf-refined")
sim_ref, gwf_ref = vh.load_valley(refined_ws)
n_ref = vh.refine_top_layer(sim_ref, target_dz, refine_depth)
vh.set_substeps(sim_ref, nstp)
add_vsf(gwf_ref, soil_name)
vh.set_lake(gwf_ref, lake_times, lake_stages)
t_ref = run(sim_ref)
print(
    f"{n_ref} refined sublayers of ~{target_dz} m: {gwf_ref.dis.nlay.array} layers; "
    f"{t_ref:4.1f} s"
)

### What the unsaturated zone looks like

Read the saturation at the end of the steady-state period and draw it along a north-south section through column 13, together with the cell edges. The base model has no saturation, so draw its cells as saturated and pass the head as `head=head`, which fills each cell only up to its head and shows the water table. In the VSF models, cells that are not fully saturated are above the water table.

In [ ]:
col_xs = 12  # column of the section, counted from 0


def end_of_steady_state(ws, name="saturation"):
    return flopy.utils.HeadFile(str(ws / "sv.sat"), text=name).get_data(idx=0)


sat_native = end_of_steady_state(native_ws)
sat_ref = end_of_steady_state(refined_ws)
# the base model is saturated below its water table, and has no saturation above
head_base = flopy.utils.HeadFile(str(base_ws / "sv.hds")).get_data(idx=0)
sat_base = np.where(head_base > 1.0e29, head_base, 1.0)

with flopy.plot.styles.USGSPlot():
    fig, axd = plt.subplot_mosaic(
        [["A"], ["B"], ["C"]],
        figsize=(width, 1.1 * width),
        layout="constrained",
        sharex=True,
    )
    for letter, gwf, sat, head, heading in [
        ("A", gwf_base, sat_base, head_base, "Base model"),
        ("B", gwf_native, sat_native, None, "VSF, native grid"),
        ("C", gwf_ref, sat_ref, None, "VSF, refined grid"),
    ]:
        ax = axd[letter]
        xs = flopy.plot.PlotCrossSection(model=gwf, line={"column": col_xs}, ax=ax)
        pc = xs.plot_array(
            np.ma.masked_greater(sat, 1.0e29),
            head=head,
            cmap="YlGnBu",
            vmin=0.4,
            vmax=1.0,
        )
        xs.plot_grid(lw=0.3, color="0.6")
        ax.set_ylim(-2.0, 20.0)
        flopy.plot.styles.heading(ax=ax, letter=letter, heading=heading)
        flopy.plot.styles.ylabel(ax=ax, label="Elevation (m)")
    flopy.plot.styles.xlabel(ax=axd["C"], label="Distance along the section (m)")
    fig.colorbar(pc, ax=list(axd.values()), shrink=0.8, label="Saturation (-)")

**What to look for.** In the base model (A) each cell is filled up to the water table and the soil above it holds no water at all. On the native grid (B) the top cell is fully saturated wherever the water table is within a meter or two of the land surface, and partly saturated only where the unsaturated zone takes up much of the cell: under the high ground and toward the south (right). Even there it has a single saturation for the whole cell and no profile. On the refined grid (C) the unsaturated zone appears everywhere, including the 1 to 2 m in the north that the native grid misses: the top cells are driest, the saturation rises downward, and it reaches 1 just above the water table. That near-saturated band is the capillary fringe. Under the high ground the water table is below the refined layers, inside the transition cell. VSF has one saturation for that cell, at its center, so it is drawn saturated up to its top.

## Step 3 &mdash; What does it change?

Compare the three models at the end of each stress period: the base model, VSF on the native grid, and VSF on the refined grid. Differences are small compared with the water table itself, so plot the **difference from the base model**. The first period is the steady state; pumping starts in the second stress period.

In the VSF models the water table is the head of the shallowest cell whose pressure head is zero or above; that cell is saturated from its center up to the water table, so its head is the water-table elevation. Interpolating the pressure head to zero between that cell and the unsaturated cell above it would put the water table too high, on the refined grid by about 9 cm on average and by up to 1.7 m: recharge moving down keeps the soil above the capillary fringe wetter than at rest, so the pressure head does not fall linearly with height.

In [ ]:
tw, wt_native = vh.read_water_tables(native_ws, vsf=True, times=period_ends)
_, wt_ref = vh.read_water_tables(refined_ws, vsf=True, times=period_ends)

year = (period_ends - period_ends[0]) / 365.25

with flopy.plot.styles.USGSPlot():
    fig, axd = plt.subplot_mosaic(
        [["A", "B"]], figsize=(width, 0.5 * width), layout="constrained"
    )
    ax = axd["A"]
    for label, wt in [("VSF, native grid", wt_native), ("VSF, refined grid", wt_ref)]:
        d = (wt - wt_base).filled(np.nan).reshape(len(year), -1)
        (line,) = ax.plot(year, np.nanmean(d, axis=1), marker="o", ms=3, label=label)
        ax.fill_between(
            year,
            np.nanpercentile(d, 5, axis=1),
            np.nanpercentile(d, 95, axis=1),
            color=line.get_color(),
            alpha=0.2,
        )
    ax.axhline(0.0, color="0.6", lw=0.6)
    flopy.plot.styles.xlabel(ax=ax, label="Years since the end of the steady state")
    flopy.plot.styles.ylabel(ax=ax, label="Water table minus base model (m)")
    flopy.plot.styles.heading(ax=ax, letter="A", heading="Whole valley, mean and 5-95%")
    flopy.plot.styles.graph_legend(ax=ax, loc="upper left")

    ax = axd["B"]
    diff = wt_ref[0] - wt_base[0]
    pmv = flopy.plot.PlotMapView(model=gwf_base, ax=ax)
    pc = pmv.plot_array(diff)
    fig.colorbar(pc, ax=ax, shrink=0.8, label="Refined minus base (m)")
    ax.set_aspect("equal")
    flopy.plot.styles.heading(ax=ax, letter="B", heading="Steady state, map")

for name, wt in [("native", wt_native), ("refined", wt_ref)]:
    d = wt - wt_base
    print(
        f"{name:8s} mean difference {d.mean():+.3f} m, largest {np.abs(d).max():.3f} m"
    )

**What to look for.** The lake is at the base model's level in both VSF models, by construction. The other differences have physical causes, which we checked with extra model runs.

- **Steady state: less water from the lake.** Away from the lake, the refined water table is up to 5 cm lower than the base model's outside the dry year (A). It is lowest on the high ground beside the lake, 5 to 6 cm lower 2 to 8 cells from the lake, and less than 1 cm lower 16 cells away (B). Recharge is the same in both models, so the water table can only be lower if less water flows through the aquifer, and the inflow that changes is the lake. It supplies 9,424 m³/d in the refined VSF model against 9,918 m³/d on the same grid without VSF, 5% less. Water leaves the lake sideways into the soil beside it and drains down from there. Where that soil is unsaturated, VSF multiplies its vertical conductance by the relative permeability, so less water gets away from the lake. Cutting the lake supply by the same amount in the model without VSF lowers the water table away from the lake by about the same amount and with the same pattern. The soil matters: with sand the lake supplies 9,110 m³/d and the water table is 4 cm lower; with clay it supplies 9,602 m³/d and the water table is 2 cm lower.
- **Dry year: the soil drains.** When the water table falls, the moisture profile above it moves down with it and the soil gives up water, up to θs − θr = 0.35 m per meter of fall for the loam, against the base model's specific yield of 0.1. With the lake held at the same level in both models, over the dry year at 13 years the refined VSF model releases 0.34 m of water per meter of water-table fall, and the base model 0.19 m, both including specific storage. The VSF water table falls less and ends the year 5 cm higher on average. The native grid drains a single thick cell per column and ends it 22 cm higher.
- **The year after: the soil refills.** When recharge returns, the soil above the water table has to be wetted again before the water table can rise. The refined VSF model stores 0.34 m of water per meter of rise, against 0.17 m for the base model, and ends the year 8 cm lower.

With the refined grid the model **simulates** the unsaturated zone.

### Water budget

The lake is handled differently in the models: in the base model its cells store and release water, and in the VSF models they are constant heads. To compare like with like, use **ZONEBUDGET** with the lake cells as a zone of their own and look at the water budget of the rest of the aquifer. `vh.aquifer_budget()` runs it on the budget file of each model and returns the inflow and outflow of every term at every time step; the flow between the aquifer and the lake is one of the terms.

Draw the budget of each year as stacked bars, as in the [`synthetic-valley-a-base`](synthetic-valley-a-base.ipynb) notebook: the inflows stack upward from zero and the outflows downward, one row per model. Each bar is the mean of the four time steps of the year, and the steady state is left out.

In [ ]:
M3D_TO_MCM_PER_YEAR = 365.25 / 1.0e6  # m³/d to million m³/yr

# budget terms, in a fixed order so each keeps its color in every row
terms = {
    "RCH": "Recharge",
    "LAKE": "Lake",
    "RIV": "Stream",
    "GHB": "Coast",
    "STO-SY": "Storage (specific yield)",
    "STO-UZ": "Storage (unsaturated zone)",
    "STO-SS": "Storage (specific storage)",
    "WEL": "Pumping wells",
    "WEL2": "Prediction well",
}
colors = dict(
    zip(
        terms,
        [
            "#2a78d6",
            "#eda100",
            "#eb6834",
            "#1baf7a",
            "#a8326e",
            "#5b2a86",
            "#e87ba4",
            "#008300",
            "#6250d6",
        ],
    )
)
models = {
    "A": ("Base model", base_ws, gwf_base),
    "B": ("VSF, native grid", native_ws, gwf_native),
    "C": ("VSF, refined grid", refined_ws, gwf_ref),
}
# the mean of the time steps of each year, leaving out the steady state
yearly = {
    letter: vh.aquifer_budget(ws, gwf).groupby("kper").mean().iloc[1:]
    * M3D_TO_MCM_PER_YEAR
    for letter, (_, ws, gwf) in models.items()
}
years = np.arange(1, len(yearly["A"]) + 1)
# room above and below the tallest stack for the Inflow and Outflow labels
tallest = max(
    budget[[f"{key}_{side}" for key in terms]].sum(axis=1).max()
    for budget in yearly.values()
    for side in ("IN", "OUT")
)
ylim = np.ceil(tallest) + 5.0

with flopy.plot.styles.USGSPlot():
    fig, axd = plt.subplot_mosaic(
        [["A"], ["B"], ["C"]],
        figsize=(width, 1.1 * width),
        layout="constrained",
        sharex=True,
    )
    handles = {}
    for letter, (heading, _, _) in models.items():
        ax = axd[letter]
        budget = yearly[letter]
        for side, sign in (("IN", 1.0), ("OUT", -1.0)):
            bottom = np.zeros(len(years))
            for key in terms:
                q = sign * budget[f"{key}_{side}"].to_numpy()
                if (q == 0).all():
                    continue
                bars = ax.bar(
                    years,
                    q,
                    bottom=bottom,
                    width=0.8,
                    color=colors[key],
                    edgecolor="white",
                    linewidth=0.5,
                )
                handles.setdefault(key, bars)
                bottom += q
        ax.axhline(0.0, color="black", lw=0.75)
        ax.set_ylim(-ylim, ylim)
        # outflows are drawn below zero, but label the axis with positive rates
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, pos: f"{abs(y):g}"))
        flopy.plot.styles.add_text(ax=ax, text="Inflow", x=0.01, y=0.97, va="top")
        flopy.plot.styles.add_text(ax=ax, text="Outflow", x=0.01, y=0.03, va="bottom")
        flopy.plot.styles.heading(ax=ax, letter=letter, heading=heading)
        flopy.plot.styles.ylabel(ax=ax, label="Rate (million m³/yr)")
    axd["C"].set_xticks(years[::2])
    flopy.plot.styles.xlabel(
        ax=axd["C"], label="Years since the end of the steady state"
    )
    # one legend entry for each term, in the order of terms
    keys = [key for key in terms if key in handles]
    flopy.plot.styles.graph_legend(
        ax=axd["C"],
        handles=[handles[key] for key in keys],
        labels=[terms[key] for key in keys],
        loc="upper center",
        bbox_to_anchor=(0.5, -0.25),
        ncol=3,
    )

**What to look for.** Recharge and pumping are the same in every model, and most years look alike: about 20 million m³/yr of recharge, plus water from the lake, leaves through the stream, the coast, and the wells. The lake supplies a little less on the refined grid, about 0.4 million m³/yr less in most years, the 5% seen in the steady state. The dry year at 13 years stands out. Recharge falls by half and storage makes up part of it: 2.3 million m³/yr in the base model, split between specific yield and specific storage, and 3.7 million m³/yr on the refined grid, mostly from the draining soil (unsaturated zone). That extra water makes up for a smaller inflow from the lake (5.2 against 6.1 million m³/yr) and keeps the discharge to the stream and the coast a little higher. The year after, the refined model puts 2.3 million m³/yr back into storage as the soil refills, against 1.5 million m³/yr, and discharges 0.6 million m³/yr less to the stream. The native grid swings further both ways.

**In plain terms.** The base model stores water only at the water table: when the water table falls 1 m, the aquifer gives up 0.1 m of water (the specific yield) and water in the soil above it is ignored. In the VSF models the soil above the water table holds water too, and how much depends on how wet it is. All of that water, at and above the water table, is counted as *Storage (unsaturated zone)*, which is why the VSF models have no specific-yield term. The soil acts as a sponge. In a dry year it drains, adds water to the aquifer, and keeps the water table and the stream up. When the rain returns, the soil has to be wetted again before recharge reaches the water table, so the aquifer and the stream recover more slowly. Over the 20 years, the water released from storage in dry years is mostly returned to storage in wet years, so the net change in storage is small: storage supplies a net 2.1 million m³ in the base model and 3.0 million m³ on the refined grid, out of 380 million m³ of recharge. The lasting difference is the lake, which supplies 5.3 million m³ less on the refined grid (69.6 against 74.9 million m³), and most of that is water that does not reach the stream (232.3 against 236.3 million m³).

## Step 4 &mdash; How fine, and how many steps?

The refinement and the time step are the two numbers the VSF model adds. Rerun the refined model with a thicker and a thinner `target_dz` and with a single time step per year, and compare the water table with the model above. This runs three more models, which takes under a minute.

In [ ]:
cases = {
    "dz 2.0 m, 4 steps": (2.0, 4),
    "dz 0.5 m, 4 steps": (0.5, 4),
    "dz 1.0 m, 1 step": (1.0, 1),
}
print(f"{'case':20s} {'run time':>9s} {'mean diff':>10s} {'largest':>8s}")
for name, (dz, steps) in cases.items():
    ws = pl.Path("models/synthetic-valley-vsf-sensitivity")
    sim, gwf = vh.load_valley(ws)
    vh.refine_top_layer(sim, dz, refine_depth)
    vh.set_substeps(sim, steps)
    add_vsf(gwf, soil_name)
    vh.set_lake(gwf, lake_times, lake_stages)
    seconds = run(sim)
    _, wt = vh.read_water_tables(ws, vsf=True, times=period_ends)
    d = wt - wt_ref
    print(f"{name:20s} {seconds:7.1f} s {d.mean():+9.3f}  {np.abs(d).max():7.3f}")

**What to look for.** The cell thickness matters little once the refined layers hold the unsaturated zone: 2 m cells are within 10 cm of the 1 m model, and 0.5 m cells within 3 cm, at about 1.7 times the run time. The time step matters more: one step per year differs by up to 23 cm. Thinner cells mostly cost run time, but too long a time step changes the answer. Check both for your own model, and check the depth of the refinement too: try `refine_depth = 2.0`, which leaves most of the unsaturated zone in the transition cell and changes the water table by up to 0.8 m.

## Recap

To put the VSF package into the valley model you:

- **held the lake** at the base model's level with constant heads, since VSF would treat its cells as soil;
- **refined the top layer** so the unsaturated zone has cells, and moved recharge, the river, the coast, and the wells onto the new layers. Without the refinement VSF lumped the unsaturated zone into one cell;
- **added the VSF package** with a soil model and its parameters, and made the cells non-convertible;
- **switched to Newton with a robust solver** and **shortened the time step**; and
- checked the **cell thickness** and **time step** against the water table.

What this notebook leaves out: the **land surface**. UZF rejects infiltration the soil cannot take and passes it on as runoff with MVR. With recharge going straight into the top cell, the VSF model has no such limit, and with the seepage drains left out, the water table rises above the land surface in some cells. A surface boundary for VSF (such as the seepage-face package VSSP) would handle that. Evapotranspiration, which the VSET package represents as root-water uptake, and a soil that varies in space, which the per-cell VSF arrays allow, are the other next steps.

## Converting your own model

Some things to check when you put VSF into an existing model. The first group stops the run; the rest change the results without a warning. VSF is still in development, so this list reflects the version used in this class and is subject to change.

**MODFLOW 6 stops with an error if**

- any active cell has no VSF soil, so every layer needs soil parameters, including deep aquifers and confining units;
- any cell is convertible (`icelltype` must be 0);
- the model uses XT3D, or the BUY or VSC packages, so a variable-density coastal model cannot use VSF; or
- the flow model is coupled to a transport (GWT) or heat (GWE) model.

**Inputs that are ignored or change meaning**

- **Specific yield and `iconvert`** are not used; the soil curves supply the drainable water. A feature represented by specific yield, such as a lake made of cells with a specific yield of 1, loses that storage. Specific storage still applies, scaled by saturation.
- **Hydraulic conductivity** still comes from NPF, including the cell-averaging option for horizontal conductance. The soil model only sets the shape of the relative permeability curve.
- **`storage_scheme` and `kr_averaging`** apply only without Newton. Under Newton, flow between two cells uses the relative permeability of the upstream cell.
- **Options for convertible cells** (THICKSTRT, VARIABLECV, DEWATERED, PERCHED, and rewetting) have no effect, because no cell converts.

**Boundaries**

- Cells under VSF never go dry, so river, general-head, and drain cells keep exchanging water with unsaturated soil, using the cell head (elevation plus pressure head). Put each one in the cell where it meets the aquifer, as `refine_top_layer()` does.
- The river still limits leakage at the riverbed bottom, as in a standard model.
- VSF multiplies the vertical conductance by the relative permeability of the upstream cell, so unsaturated soil passes much less water down than a convertible cell under NPF, whose vertical conductance does not depend on how wet the cell is.
- `AUTO_FLOW_REDUCE` on a well does nothing in VSF cells, so a well in drying soil keeps pumping at its full rate.
- UZF, EVT, and land-surface drains do the jobs that the VSF surface packages (VSSB, VSSP, and VSET) are built for. Decide which to keep before converting.

**Grid, solver, and output**

- The refined layers must hold the unsaturated zone where the water table is deepest, not just on average.
- Pass-through cells (`idomain` of -1) work as in a standard model, because VSF uses only the active cells. Replacing the valley's pass-through cells with 1 cm active cells changes the water table by less than 1 mm. The output files hold 1e30 in those cells, so skip them when looking for the water table, as `water_table()` does.
- Open water built from very permeable cells, like the lake here, becomes soil under VSF, and its very large conductivity in thin cells can stop the solver from converging. Hold it with constant heads, as `set_lake()` does.
- Splitting a layer keeps its anisotropy as long as every sublayer gets the layer's horizontal and vertical conductivity, as here, where the vertical conductivity is 0.4 times the horizontal everywhere. If the vertical conductivity of a layer stands for a low-permeability bed at its base, put that bed in the bottom sublayer instead.
- The steady state converges much less reliably than in a standard model, and changes in the grid that look minor can decide whether it converges. Expect to adjust the solver and the starting heads.
- Annual time steps are too long; check the time step as in the last section.
- The head in an unsaturated cell is not the water table. Head observations, calibration targets, and head maps have to find the water table from the pressure head, as `water_table()` does.
- The budget reports VSF storage as `STO-UZ` rather than `STO-SS` and `STO-SY`, so scripts that read the budget need updating.
